<a href="https://colab.research.google.com/github/Danielnodn/academic-ml-projects/blob/main/feature_engeneering/california_housing/V1_%20cali_housing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# !pip install pandas numpy boto3 scikit-learn pyyaml

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import boto3
import yaml

from io import StringIO
from datetime import datetime, date

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:

import boto3
import yaml
from io import StringIO

def load_data_s3(bucket_name, object_key, credentials_path):

    """

    with open(credentials_path, 'r') as file:
        creds = yaml.safe_load(file)

    s3 = boto3.client(
        's3',
        aws_access_key_id=creds['aws_access_key_id'],
        aws_secret_access_key=creds['aws_secret_access_key']
    )

    obj = s3.get_object(Bucket=bucket_name, Key=object_key)
    data = obj['Body'].read().decode('utf-8')

    return pd.read_csv(StringIO(data))

"\n# import boto3\n# import yaml\n# from io import StringIO\n\n# def load_data_s3(bucket, key, credentials_path):\n#     with open(credentials_path, 'r') as file:\n#         creds = yaml.safe_load(file)\n\n#     s3 = boto3.client(\n#         's3',\n#         aws_access_key_id=creds['aws_access_key_id'],\n#         aws_secret_access_key=creds['aws_secret_access_key']\n#     )\n\n#     obj = s3.get_object(Bucket=bucket, Key=key)\n#     data = obj['Body'].read().decode('utf-8')\n#     return pd.read_csv(StringIO(data))\n"

In [ ]:
# Abriendo sesión s3
with open('credentials.yaml', 'r') as f:
  config = yaml.safe_load(f)

In [ ]:
def carga_datos_s3(bucket, bucket_path):
  session = boto3.Session(
      aws_access_key_id= config['s3']['aws_access_key_id'],
      aws_secret_access_key= config['s3']['aws_secret_access_key'],
      aws_session_token= config['s3']['aws_session_token']
  )

  s3 = session.resource('s3')

  obj = s3.Object(bucket, bucket_path).get()['Body'].read()
  dataset = pickle.loads(obj)

  return dataset

In [ ]:
session = boto3.Session(
      aws_access_key_id= config['s3']['aws_access_key_id'],
      aws_secret_access_key= config['s3']['aws_secret_access_key'],
      aws_session_token= config['s3']['aws_session_token']
      )
s3 = session.client('s3')

In [ ]:
s3_client = boto3.client('s3')
bucket_name= 'ap-cd-1-mcda23b043'

In [ ]:
bucket = bucket_name #+ config['iexe']['matricula']
key='limpieza/'

In [ ]:
bucket_path = s3.list_objects_v2(Bucket = bucket, Prefix = key)['Contents'][0]['Key']
bucket_path

In [ ]:
dataset = carga_datos_s3(bucket, bucket_path)

In [ ]:
# transformar ingesta

def transformar_ingesta (dataframe):
  df=pd.DataFrame(dataframe)
  df = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)

  return df



In [ ]:
inspecciones = transformar_ingesta(dataset)

In [ ]:
inspecciones.shape

In [ ]:
inspecciones.head()

In [ ]:
def delete_inspections(dataframe):
    un_results = ["business not located", "no entry", "out of business"]

    cleaned_df = dataframe[~dataframe['results'].isin(un_results)]

    return cleaned_df


inspecciones_limpias = delete_inspections(inspecciones)


In [ ]:
inspecciones_limpias

In [ ]:
# ¿Cuatos registros de inspecciones se tiene ahora?
inspecciones_limpias.shape

In [ ]:
def unique_results(dataframe):
    # Obtener los valores únicos de la columna 'results'
    unique_values = dataframe['results'].unique()

    return unique_values

# Ejemplo de uso:
valores_unicos_results = unique_results(inspecciones_limpias)
print("Valores únicos de la columna 'results':", valores_unicos_results)


In [ ]:
# 3. Transformación de la columna result
def transform_result(dataframe):

    result_mapping = {
        'pass': 'pass',
        'pass w/ conditions': 'pass',
        'not ready': 'fail',
        'fail':'fail'
        }


    dataframe['results'] = dataframe['results'].map(result_mapping).fillna('fail')

    return dataframe


inspecciones_transformadas = transform_result(inspecciones_limpias)




In [ ]:
# Resultados pass y fail
inspecciones_transformadas['results'].value_counts()


In [ ]:
#4.  Transformación de columna risk
def transform_risk(dataframe):

    risk_mapping = {
        'risk 1 (high)': 'high',
        'risk 2 (medium)': 'medium',
        'risk 3 (low)': 'low',

    }


    dataframe['risk'] = dataframe['risk'].map(risk_mapping).fillna('all')

    return dataframe


inspecciones_transformadas = transform_risk(inspecciones_transformadas)



In [ ]:
# conteo de las variables en columna risk
inspecciones_transformadas['risk'].value_counts()

In [ ]:
# 5. Transformación de facility_type
def transform_facility(df):
    df['facility_type'] = df['facility_type'].str.lower()
    df.loc[df['facility_type'].str.contains('daycare'), 'facility_type'] = 'daycare'
    df.loc[df['facility_type'].str.contains('restaurant'), 'facility_type'] = 'restaurant'
    df.loc[df['facility_type'].str.contains('mobile food'), 'facility_type'] = 'mobile food'

    return df

In [ ]:
def transform_facility_other(df):
    # Obtener el top 20 de facility_type con más registros
    top_20 = df['facility_type'].value_counts().head(20).index

    # Cambiar los facility_type que no estén en el top 20 a 'other'
    df.loc[~df['facility_type'].isin(top_20), 'facility_type'] = 'other'

    return df

In [ ]:
inspecciones = inspecciones.dropna(subset=['facility_type'])

In [ ]:
inspecciones_transformadas = transform_facility(inspecciones)

In [ ]:
inspecciones_transformadas = transform_facility_other(inspecciones_transformadas)

In [ ]:
top_20_facility_types

In [ ]:
top_30_facility_types = inspecciones_transformadas['facility_type'].value_counts().head(30)
print(top_30_facility_types)

In [ ]:
inspecciones_transformadas['facility_type'].value_counts()

In [ ]:
# 6 generar features

from datetime import datetime

def generate_features(dataframe):

    dataframe['inspection_date'] = pd.to_datetime(dataframe['inspection_date'])


    dataframe['month'] = dataframe['inspection_date'].dt.month
    dataframe['year'] = dataframe['inspection_date'].dt.year
    dataframe['day_of_month'] = dataframe['inspection_date'].dt.day
    dataframe['week_of_year'] = dataframe['inspection_date'].dt.isocalendar().week
    dataframe['day_of_week'] = dataframe['inspection_date'].dt.dayofweek


    dataframe['week_day'] = dataframe['day_of_week'].apply(lambda x: 1 if x < 5 else 0)
    dataframe['weekend'] = dataframe['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)

    return dataframe


In [ ]:
inspecciones_con_features = generate_features(inspecciones_transformadas)

In [ ]:
# Mostrar features
inspecciones_con_features.head()

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

def feature_matrix(dataframe):

    df = dataframe[['facility_type', 'risk', 'latitude', 'longitude',
                    'month', 'year', 'day_of_month', 'week_of_year',
                    'day_of_week', 'week_day', 'weekend']]


    transformers = [
        ('one_hot_risk', OneHotEncoder(), ['risk']),
        ('one_hot_facility_type', OneHotEncoder(), ['facility_type'])
    ]


    ct = ColumnTransformer(transformers, remainder="passthrough", n_jobs=-1)


    feature_matrix = ct.fit_transform(df)


    feature_names = np.array(ct.get_feature_names_out())


    feature_matrix = pd.DataFrame(feature_matrix, columns=feature_names)

    return feature_matrix

In [ ]:
matriz_de_caracteristicas = feature_matrix(inspecciones_con_features)


In [ ]:
print("Forma de la matriz de características:", matriz_de_caracteristicas.shape)
matriz_de_caracteristicas.head()

In [ ]:
# 6 guardar matriz
from datetime import date

def save_feature_matrix(bucket, bucket_path, dataset):
    session = boto3.Session(
        aws_access_key_id = config['s3']['aws_access_key_id'],
        aws_secret_access_key = config['s3']['aws_secret_access_key'],
        aws_session_token = config['s3']['aws_session_token']
    )

    s3 = session.resource('s3')

    s3.Object(bucket, bucket_path).put(Body=dataset)

# Obtener la fecha actual
TODAY = date.today()

In [ ]:
pickle_data = pickle.dumps(matriz_de_caracteristicas)

In [ ]:
bucket = 'ap-cd-1-mcda23b043'
key = 'feature-matrix/feature-matrix-' + str(TODAY) + '.pkl'

In [ ]:
save_feature_matrix(bucket, key, pickle_data)
